# Microservice Cascade Experiments

Run experiments and figures for the risk propagation / failure containment simulator. Use **Google Colab** (CPU) or local. Run all cells in order.

## 1. Install dependencies

In [ ]:
!pip install -q numpy pandas networkx matplotlib pyyaml tqdm

## 2. Clone repo (Colab) or set project root

In Colab: clone the repo and `cd` into it. If you already uploaded the project, set `PROJECT_DIR` to its path.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/code-ninja-bayyavarapu/Dynamic-Risk-Propagation-and-Failure-Containment-in-Large-Scale-Enterprise-Microservice-Graphs.git"
PROJECT_NAME = "Dynamic-Risk-Propagation-and-Failure-Containment-in-Large-Scale-Enterprise-Microservice-Graphs"

if Path("/content").exists():
    # Colab: ensure we have the repo and get latest
    repo_path = Path("/content") / PROJECT_NAME
    if (Path(".").resolve() / "risk_containment").exists():
        # Already in repo root (e.g. opened from GitHub)
        os.chdir(Path(".").resolve())
        get_ipython().system("git pull origin master")
    elif repo_path.exists():
        os.chdir(repo_path)
        get_ipython().system("git pull origin master")
    else:
        get_ipython().system(f"git clone {REPO_URL}")
        os.chdir(repo_path)

PROJECT_DIR = Path(".").resolve()
print("Project root:", PROJECT_DIR)

## 3. Run experiments

Runs the simulator with graph sizes 50, 200, 500 (or reduced for speed), 20 seeds (or fewer for quick run), 4 scenarios, 5 strategies. Uses deterministic seeds for reproducibility.

**Quick run (Colab-friendly):** 50 nodes, 2 seeds, ~40 runs, ~2–3 min.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

# Quick run (Colab-friendly): 50 nodes, 2 seeds, realistic simulator
get_ipython().system("python scripts/run_experiments.py --use-simulator --seeds 2 --sizes 50 --output-dir outputs/experiments")

## 4. Generate figures

Produces SLA vs strategy, cascade peak vs strategy, throughput comparison, MTTR comparison, containment vs SLA. Saves to `paper/figures/` and generates `paper/results_table.tex`.

In [ ]:
!python scripts/generate_figures.py

## 5. Display results

In [ ]:
import pandas as pd
from pathlib import Path

summary_path = Path("outputs/experiments/summary.csv")
if summary_path.exists():
    df = pd.read_csv(summary_path)
    display(df.head(20))
    print("Shape:", df.shape)
    print("\nMean metrics by strategy:")
    display(df.groupby("strategy")[["cascade_size_peak", "sla_violations", "throughput_completed", "mttr"]].mean())
else:
    print("No summary.csv found. Run the experiment cell first.")

In [ ]:
from IPython.display import display, Image, Markdown

fig_dir = Path("paper/figures")
if fig_dir.exists():
    for name in ["cascade_comparison", "sla_vs_strategy", "throughput_comparison", "mttr_comparison", "containment_vs_sla"]:
        p = fig_dir / f"{name}.png"
        if p.exists():
            display(Markdown(f"### {name}"))
            display(Image(filename=str(p), width=500))
else:
    print("No paper/figures. Run generate_figures.py first.")

## 6. Export CSV and plots

Download or inspect paths for CSV and figures.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

out = Path("outputs/experiments")
paper_fig = Path("paper/figures")

print("Output files:")
for f in [out / "summary.csv", out / "per_run_metrics.csv", out / "cascade_metrics.csv", out / "throughput_metrics.csv"]:
    if f.exists():
        print("  ", f)
for f in paper_fig.glob("*.png") if paper_fig.exists() else []:
    print("  ", f)

# Uncomment to trigger download in Colab:
# files.download("outputs/experiments/summary.csv")
# files.download("paper/figures/cascade_comparison.png")